# Step 1: Getting training working

This is pretty janky, having issues getting training working. If anyone get it going or finds improvements to this (in terms of execution), please comment!

In [ ]:
import polars as pl
import os
import gc
import torch
import mamba_ssm
import kagglehub
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset, DataLoader

SUBSAMPLE_SIZE = 600

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(f"Training samples: {len(train)}")
print(train.head())
train = train.sample(n=SUBSAMPLE_SIZE, seed=42)

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import shutil, os, stat, sys

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
shutil.copy2(src, dst)
os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

# Copy the entire bin dir so Triton can find everything
import triton.backends.nvidia as nv_backend
src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
dst_bin = "/tmp/triton_nvidia_bin"
shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
for f in os.listdir(dst_bin):
    fp = os.path.join(dst_bin, f)
    if os.path.isfile(fp):
        os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

# Redirect Triton's path resolution to the writable copy
nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
os.environ["TRITON_PTXAS_PATH"] = dst

LORA_RANK = 32 # test others
MAX_SEQ_LEN = 1024 # raise later
NUM_EPOCHS = 1 # for getting it working, raise later
GRAD_ACCUM = 4 # raise later
LR = 2e-4 # tune

# ── Load model ONCE ──
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Force slow path — bypass the broken CUDA kernels
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded. Vocab size: {len(tokenizer)}")

# ── LoRA ──
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Enable gradient checkpointing with use_reentrant=False ──
# use_reentrant=False avoids the tensor-count mismatch with our patched slow path
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

def build_training_text(prompt: str, answer: str) -> str:
    user_msg = prompt + "\nPlease put your final answer inside \\boxed{}."
    assistant_msg = f"\\boxed{{{answer}}}"

    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )
    return text

texts = [
    build_training_text(row["prompt"], row["answer"])
    for row in train.iter_rows(named=True)
]
print(f"Built {len(texts)} training examples")
print(f"Example:\n{texts[0][:500]}")

class SFTDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.encodings = []
        skipped = 0
        for text in texts:
            enc = tokenizer(
                text, truncation=True, max_length=max_length,
                padding="max_length", return_tensors="pt",
            )
            ids = enc["input_ids"].squeeze(0)
            mask = enc["attention_mask"].squeeze(0)

            labels = ids.clone()
            labels[mask == 0] = -100

            self.encodings.append({
                "input_ids": ids,
                "attention_mask": mask,
                "labels": labels,
            })

        print(f"Tokenized {len(self.encodings)} examples (skipped {skipped})")

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return self.encodings[idx]

dataset = SFTDataset(texts, tokenizer, MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
import torch, torch.nn.functional as F, sys

# ── Bypass Triton rmsnorm (avoids ptxas-blackwell permission issue) ──
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# ── Training loop ──
model.train()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=0.01,
)

total_steps = (len(dataloader) * NUM_EPOCHS) // GRAD_ACCUM
print(f"Training: {NUM_EPOCHS} epochs, ~{total_steps} optimizer steps, "
      f"{len(dataset)} examples")

for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    optimizer.zero_grad()

    for i, batch in enumerate(dataloader):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        running_loss += outputs.loss.item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

            step = (i + 1) // GRAD_ACCUM
            if step % 20 == 0:
                avg = running_loss / (i + 1)
                print(f"  epoch {epoch+1} | step {step} | avg_loss {avg:.4f}")

    if (i + 1) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} — avg loss: {avg_loss:.4f}")

In [ ]:
import zipfile

model.save_pretrained(OUTPUT_DIR)
print(f"Adapter files saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

# Package into submission.zip (required by competition)
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)  # files at zip root

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

# Verify it contains adapter_config.json (required)
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("✓ submission.zip looks correct")